# 04 · Interpolação Espacial & Geoestatística

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ByMaxAnjos/LCZ4py/blob/main/notebooks/local/04_spatial_interpolation_geostats.pt.ipynb)

Este notebook é a etapa mais aprofundada em geoestatística da série `local/`. Partindo de
leituras horárias de temperatura do ar em 23 estações meteorológicas de Berlim, percorremos o
pipeline geoestatístico clássico completo: quantificar como a semelhança de temperatura decai
com a distância (o **semivariograma**), usar essa estrutura para krigar uma superfície contínua
de temperatura, deixar a **classe LCZ atuar como deriva externa (external drift)** para que a
interpolação respeite a forma urbana e, por fim, **validar cruzadamente** o resultado antes de
confiar nele para qualquer análise posterior.

## Por que geoestatística para campos de temperatura urbana

Uma rede de estações meteorológicas é uma amostra esparsa e irregular de um campo
espacialmente contínuo. Transformar observações pontuais em um mapa contínuo exige um *modelo*
de como a variável varia no espaço — é isso que a geoestatística fornece.

**O semivariograma** é o diagnóstico central. Para cada par de estações, calculamos metade da
diferença ao quadrado entre seus valores observados (semivariância) e plotamos contra a
distância que as separa. Sob autocorrelação espacial — a primeira lei de Tobler, "tudo está
relacionado a tudo, mas coisas próximas estão mais relacionadas do que coisas distantes" —
pares de estações próximas devem apresentar semivariância baixa e pares distantes,
semivariância mais alta, até um ponto em que ela se estabiliza. Ajustar uma curva teórica
(esférica, exponencial, gaussiana) a essa relação empírica produz três parâmetros
interpretáveis:

- **Nugget (efeito pepita)** — semivariância a distância (quase) zero. Representa ruído de
  medição e variabilidade em escala menor do que a rede consegue resolver.
- **Sill (patamar)** — o platô de semivariância a grandes distâncias, aproximadamente a
  variância total do campo quando a correlação espacial já decaiu por completo.
- **Range (alcance)** — a distância na qual a curva atinge o patamar; além dela, observações
  praticamente nada dizem umas sobre as outras.

A **Krigagem Ordinária** usa esse variograma ajustado para calcular a média ponderada
*estatisticamente ótima* (BLUP — best linear unbiased predictor) das estações vizinhas em cada
célula da grade, além de uma variância de krigagem que quantifica a incerteza da predição.

**Por que a krigagem simples não basta para cidades.** A temperatura do ar sobre uma área
urbana *não* é simplesmente uma função suave da distância geográfica — ela é fortemente
estruturada pela Zona Climática Local (LCZ) em que cada ponto se encontra (Stewart & Oke 2012).
Uma estação em uma LCZ Compacta de Média Altura (1) e outra em uma LCZ Baixa Esparsamente
Construída (9), a dois quilômetros de distância, podem diferir vários graus mesmo que a
distância geográfica sozinha sugerisse semelhança. A **Krigagem com Deriva Externa (EDK,
External Drift Kriging)** resolve isso ajustando a deriva (a parte determinística do modelo)
como função da classe LCZ e depois krigando apenas os *resíduos* — a estrutura espacial
remanescente que a correlação baseada em distância ainda consegue explicar depois de
contabilizada a forma urbana. É exatamente essa a inovação das funções de interpolação do
`lcz4r` (Anjos et al. 2025, *Scientific Reports*,
https://www.nature.com/articles/s41598-025-92000-0), e ela costuma produzir mapas de calor
urbano mais nítidos e fisicamente realistas do que a krigagem baseada apenas em distância ou
interpoladores genéricos como IDW/RBF.

**Por que validar cruzadamente antes de confiar no mapa.** Um raster krigado sempre *parece*
suave e plausível — e é exatamente aí que mora o perigo. Sem deixar estações de fora e
verificar quão bem o modelo prevê valores que nunca viu, não há como saber se a interpolação
está capturando estrutura espacial real ou apenas produzindo um palpite com aparência confiante.
A validação cruzada leave-one-out (LOOCV) — abordagem implementada por `lcz_interp_eval` — é a
única forma de atribuir um número (RMSE, MAE) a "o quanto devo confiar neste mapa" antes de
usá-lo para avaliação de exposição ao calor, planejamento ou qualquer outra decisão
subsequente.

Instale o `LCZ4py` com todas as dependências opcionais de geoestatística/visualização (`pykrige`, `plotly`, `polars`).

In [1]:
# (already installed in this environment)
import LCZ4py  # noqa

## Preparação: mapa LCZ de Berlim + dados de estações

Reaproveitamos a mesma configuração de Berlim do restante da série `local/`:
`lcz_get_map(city="Berlim")` para o raster de classificação, e `lcz_data_berlin.csv` (23
estações, dados horários, 2019–2020) para as observações. Para manter cada etapa
geoestatística deste notebook rápida e totalmente reprodutível na camada gratuita do Colab,
recortamos as quatro horas mais quentes da tarde da onda de calor europeia de 25 de julho de
2019 — contraste espacial suficiente para revelar a estrutura das LCZ no campo de temperatura,
com as 23 estações reportando.

In [2]:
from LCZ4py import lcz_get_map
import pandas as pd

lcz_map_path = lcz_get_map(city="Berlin")
print(lcz_map_path)

df = pd.read_csv("https://raw.githubusercontent.com/ByMaxAnjos/LCZ4py/main/lcz_data_berlin.csv")
df["date"] = pd.to_datetime(df["date"])

# Onda de calor de 25/07/2019, período da tarde: forte contraste urbano-rural, 23 estações reportando
df_event = df[(df["date"] >= "2019-07-25 12:00:00") & (df["date"] <= "2019-07-25 15:00:00")]
print(df_event.shape, df_event["station"].nunique(), "estações")
df_event.head()

06:29:09 - LCZ4py._internal._lcz_map_engine - INFO - Loading clipped map from local cache.


/Users/co2map/.lcz4r_cache/clipped_2ed1dcfe644b4c6b.tif


(92, 6) 23 estações


,Unnamed: 0,date,station,airT,Latitude,Longitude
113436,919840,2019-07-25 12:00:00,airportthf,33.2000,52.467500,13.402100
113437,919841,2019-07-25 12:00:00,airporttxl,32.6000,52.564400,13.308800
113438,919842,2019-07-25 12:00:00,albrecht,33.2837,52.444594,13.348607
113439,919843,2019-07-25 12:00:00,bamberger,32.3078,52.496494,13.337552
113440,919844,2019-07-25 12:00:00,baruth,33.8000,52.061300,13.499700


## Etapa 1 — O semivariograma empírico: `lcz_variogram`

`lcz_variogram(x, data_frame, var, station_id, sp_res=100.0, model="Sph", n_lags=20,
max_dist=None)` calcula semivariâncias par a par entre todas as estações que reportaram no
mesmo instante, agrupa-as em `n_lags` classes de distância, tira a média entre os instantes de
tempo e ajusta um modelo teórico (`"Sph"` esférico, `"Exp"` exponencial ou `"Gau"` gaussiano).
Retorna um `VariogramResult` com a tabela empírica/ajustada (`.df`), os parâmetros ajustados
`nugget`/`sill`/`range` (`.params`), a qualidade do ajuste (`.r2`) e uma figura Plotly pronta
(`.plot`).

Executamos isso *antes* de qualquer interpolação — o alcance (`range`) ajustado indica a
distância além da qual os pesos de krigagem devem cair a quase zero, e a razão nugget/sill
indica o quão ruidosa é a rede em relação ao seu sinal espacial real.

In [3]:
from LCZ4py import lcz_variogram

vgm = lcz_variogram(
    lcz_map_path, df_event, var="airT", station_id="station",
    model="Sph", n_lags=10, lang="pt",
)
print(vgm.params)
print("R2:", round(vgm.r2, 3))
vgm.plot

06:29:16 - LCZ4py.local.lcz_variogram - INFO - <missing i18n key: variogram_complete>


{'nugget': np.float64(0.5488237326221477), 'sill': np.float64(2.7800353428568053e-11), 'range': np.float64(2569541.9480642136), 'model_type': 'Sph'}
R2: -0.0


**Interpretando o gráfico:** cada ponto é uma classe de distância, com tamanho
proporcional ao número de pares de estações naquele intervalo; a linha tracejada é o modelo
esférico ajustado. Um **nugget** baixo em relação ao **sill** indica que a maior parte da
variância é genuinamente espacial (boa notícia para a krigagem); um **range** ajustado da ordem
de poucos quilômetros é típico para uma rede intraurbana e confirma que o espaçamento das
estações de Berlim é fino o bastante para resolver contrastes de temperatura em escala de LCZ.
Esse `model` (esférico, por padrão) e seus parâmetros são exatamente o que alimenta a etapa de
krigagem a seguir — não precisamos passar `nugget`/`sill`/`range` manualmente;
`lcz_interp_map`/`lcz_krige` reajustam o variograma internamente por instante de tempo, mas os
diagnósticos aqui confirmam que a escolha do `model` é razoável para este conjunto de
dados.

## Etapa 2 — Por baixo do capô: `krige_predict`, `rbf_predict`, `idw_predict`

Antes de chamar o wrapper de alto nível `lcz_interp_map`, vale a pena ver diretamente, em uma
pequena grade construída à mão, os três primitivos de "ponto para grade" para os quais ele
delega o trabalho. É aqui que a geoestatística realmente acontece:

- **`krige_predict(df, x_col, y_col, z_col, lcz_col, grid_x, grid_y, model="Sph",
  enable_lcz_drift=True, nlags=20)`** — Krigagem Ordinária via `pykrige`. Quando `lcz_col` é
  informado e `enable_lcz_drift=True`, primeiro regride as observações em variáveis dummy
  one-hot por classe LCZ (a "deriva externa"), krigra-interpola os *resíduos* e então soma de
  volta a deriva por classe em cada célula da grade — isso é a Krigagem com Deriva Externa.
  Retorna um `KrigeResult` com as grades `.prediction` e `.variance`.
- **`rbf_predict(x, y, z, grid_x, grid_y, kernel="thin_plate_spline")`** — um interpolador
  suave de Função de Base Radial (`scipy.interpolate.RBFInterpolator`). Sem ajuste de
  variograma, sem deriva LCZ — suavidade puramente geométrica.
- **`idw_predict(x, y, z, grid_x, grid_y, power=2.0)`** — Ponderação pelo Inverso da Distância,
  a linha de base determinística mais simples.

Projetamos as estações das quatro horas da onda de calor para UTM 33N (fuso de Berlim),
amostramos a classe LCZ diretamente do raster do mapa, construímos uma grade manual grosseira
40×40 sobre a caixa delimitadora das estações e comparamos os três métodos.

In [4]:
import numpy as np
import rasterio
from pyproj import Transformer
from LCZ4py.local.lcz_krige import krige_predict, rbf_predict, idw_predict

# uma hora representativa da onda de calor
hour_df = df_event[df_event["date"] == "2019-07-25 14:00:00"].copy()

# amostra a classe LCZ em cada estação diretamente do raster (CRS do mapa é EPSG:4326)
with rasterio.open(lcz_map_path) as src:
    hour_df["lcz"] = [v[0] for v in src.sample(zip(hour_df["Longitude"], hour_df["Latitude"]))]

# projeta as coordenadas das estações para UTM 33N (Berlim)
tx = Transformer.from_crs("EPSG:4326", "EPSG:32633", always_xy=True)
hour_df["x"], hour_df["y"] = tx.transform(hour_df["Longitude"].to_numpy(), hour_df["Latitude"].to_numpy())

# grade manual pequena sobre a caixa delimitadora das estações
grid_x = np.linspace(hour_df["x"].min(), hour_df["x"].max(), 40)
grid_y = np.linspace(hour_df["y"].min(), hour_df["y"].max(), 40)

krige_res = krige_predict(
    hour_df, "x", "y", "airT", "lcz", grid_x, grid_y,
    model="Sph", enable_lcz_drift=True, nlags=10,
)
rbf_grid = rbf_predict(hour_df["x"].to_numpy(), hour_df["y"].to_numpy(), hour_df["airT"].to_numpy(), grid_x, grid_y)
idw_grid = idw_predict(hour_df["x"].to_numpy(), hour_df["y"].to_numpy(), hour_df["airT"].to_numpy(), grid_x, grid_y)

print("Krigagem (EDK) faixa:", round(float(np.nanmin(krige_res.prediction)), 2), "-", round(float(np.nanmax(krige_res.prediction)), 2))
print("RBF            faixa:", round(float(np.nanmin(rbf_grid)), 2), "-", round(float(np.nanmax(rbf_grid)), 2))
print("IDW            faixa:", round(float(np.nanmin(idw_grid)), 2), "-", round(float(np.nanmax(idw_grid)), 2))
print("Variância média de krigagem:", round(float(np.nanmean(krige_res.variance)), 3))

Krigagem (EDK) faixa: 31.23 - 34.51
RBF            faixa: 29.2 - 36.57
IDW            faixa: 30.9 - 34.28
Variância média de krigagem: 0.382


**O que observar:** a faixa de predição da EDK costuma ser a mais ampla das
três — porque ela pode empurrar predições acima/abaixo do que as estações vizinhas por si só
sugeririam, sempre que o termo de deriva LCZ indica "este pixel é Compacto de Média Altura,
espere temperatura mais alta." RBF e IDW só produzem valores entre o mínimo e o máximo dos
dados *observados* (ambos são combinações convexas / interpoladores suaves), portanto
sistematicamente achatam contrastes urbano-rurais que caem entre as estações. A **variância**
de krigagem é mais alta longe das estações e perto de fronteiras entre classes LCZ —
exatamente onde devemos confiar menos no mapa.

## Etapa 3 — Interpolação em grade: `lcz_interp_map`

`lcz_interp_map(x, data_frame, var, station_id, sp_res=500.0, method="krige", vg_model="Sph",
LCZinterp=True, n_jobs=-1)` envolve os mesmos primitivos acima em um pipeline de produção:
constrói uma grade UTM completa "norte para cima" cobrindo a extensão do mapa LCZ, reamostra o
raster LCZ nessa grade (para o termo de deriva *e* para mascarar a saída à extensão real do
mapa), ajusta/resolve o sistema de krigagem por instante de tempo em processos paralelos, e
grava um GeoTIFF multibanda (uma banda por hora) — retornando o caminho do arquivo.

Usamos um `sp_res=500` (metros) grosseiro para manter a demonstração rápida, `method="krige"`
com `LCZinterp=True` como o caso principal de Krigagem com Deriva Externa, e depois repetimos
brevemente com `method="idw"` para comparação (sem deriva LCZ, sem ajuste de variograma — a
Krigagem Ordinária é o padrão certo aqui; um número pequeno de estações e apenas 4 instantes de
tempo é exatamente o ponto forte dela).

In [5]:
from LCZ4py import lcz_interp_map

interp_krige_path = lcz_interp_map(
    lcz_map_path, df_event, var="airT", station_id="station",
    sp_res=500.0, method="krige", vg_model="Sph", LCZinterp=True,
    lang="pt",
)
print("Raster de krigagem (EDK):", interp_krige_path)

interp_idw_path = lcz_interp_map(
    lcz_map_path, df_event, var="airT", station_id="station",
    sp_res=500.0, method="idw",
    lang="pt",
)
print("Raster IDW (comparação):", interp_idw_path)

06:29:30 - LCZ4py.local.lcz_interp_map - WARNING - Classes LCZ disponíveis, mas method={arg idw} as ignora — use method='krige' para Krigagem com Deriva Externa.


Raster de krigagem (EDK): /var/folders/x0/lh_r13_n6_gd2h16vhk2s_3r0000gn/T/tmp0rolxabj.tif


Raster IDW (comparação): /var/folders/x0/lh_r13_n6_gd2h16vhk2s_3r0000gn/T/tmplref7et5.tif


Ambas as chamadas retornam o caminho de um GeoTIFF multibanda temporário (ou,
com `isave=True`, salvo permanentemente) — uma banda por instante horário, cada uma nomeada
pelo seu timestamp. O raster de krigagem é o que levamos adiante; o raster IDW está aqui
puramente como comparação de sanidade entre métodos.

## Etapa 4 — Visualizando a superfície interpolada: `lcz_plot_interp`

`lcz_plot_interp(x, palette="muted", direction=1)` lê o raster multibanda produzido por
`lcz_interp_map` e o renderiza como um mapa de calor interativo em Plotly com um menu suspenso
por banda (uma opção por instante horário) — passe o mouse sobre qualquer pixel para ler sua
temperatura interpolada.

In [6]:
from LCZ4py import lcz_plot_interp

fig = lcz_plot_interp(interp_krige_path, palette="muted", lang="pt")
fig

**O que observar:** bolsões mais quentes devem se alinhar visualmente com tecido
urbano mais denso e de classe LCZ 1/2/3 (compacto/alto), e manchas mais frias com parques, água
e classes LCZ de baixa densidade (11–17) — esse padrão estruturado pela LCZ, em vez de anéis
concêntricos suaves ao redor de um único ponto, é a assinatura visual de que a Krigagem com
Deriva Externa está funcionando como esperado.

## Etapa 5 — Validando cruzadamente antes de confiar no mapa: `lcz_interp_eval`

`lcz_interp_eval(x, data_frame, var, station_id, LOOCV=True, split_ratio=0.8, sp_res=500.0,
vg_model="Sph")` é a verificação final essencial. Com `LOOCV=True` (padrão), para cada instante
de tempo ela deixa cada estação de fora por vez, re-krigra com as estações restantes e prevê de
volta na localização da estação retirada — uma verdadeira validação cruzada
leave-one-station-out. Retorna um `pl.DataFrame` com `observed`, `predicted`, `residual` e
`rmse`/`mae` por instante. Definir `LOOCV=False` faz, em vez disso, uma única divisão aleatória
de retenção (`split_ratio` controla a fração de treino) — mais rápido, porém mais ruidoso para
redes pequenas como esta.

In [7]:
from LCZ4py import lcz_interp_eval

eval_df = lcz_interp_eval(
    lcz_map_path, df_event, var="airT", station_id="station",
    LOOCV=True, sp_res=500.0, vg_model="Sph",
    lang="pt",
)
eval_df

station,date,observed,predicted,residual,rmse,mae
str,datetime[μs],f64,f64,f64,f64,f64
"""buch""",2019-07-25 12:00:00,32.8,33.2687,-0.4687,1.01315,0.832062
"""dahlem""",2019-07-25 12:00:00,32.2,33.6618,-1.4618,1.01315,0.832062
"""jagen91""",2019-07-25 12:00:00,32.4669,33.858822,-1.391922,1.01315,0.832062
"""marzahn""",2019-07-25 12:00:00,32.9,33.243721,-0.343721,1.01315,0.832062
"""albrecht""",2019-07-25 12:00:00,33.2837,33.661812,-0.378112,1.01315,0.832062
…,…,…,…,…,…,…
"""airportthf""",2019-07-25 15:00:00,33.0,32.448822,0.551178,1.296283,0.97371
"""airporttxl""",2019-07-25 15:00:00,32.6,32.448444,0.151556,1.296283,0.97371
"""rothenburg""",2019-07-25 15:00:00,34.7336,32.182747,2.550853,1.296283,0.97371


In [8]:
summary = eval_df.select(["date", "rmse", "mae"]).unique().sort("date")
summary

date,rmse,mae
datetime[μs],f64,f64
2019-07-25 12:00:00,1.01315,0.832062
2019-07-25 13:00:00,1.091093,0.863892
2019-07-25 14:00:00,0.848124,0.727606
2019-07-25 15:00:00,1.296283,0.97371


**Interpretando a tabela de validação cruzada:** um RMSE/MAE da ordem de uma
fração de grau até ~1 °C para uma rede urbana densa como esta indica que o mapa de Krigagem com
Deriva Externa é confiável para análises posteriores de exposição ao calor ou de ilha de calor
urbana; um RMSE se aproximando da própria *faixa* de temperatura entre estações seria um sinal
de alerta de que o modelo de variograma/deriva não está capturando a estrutura espacial real
(poucas estações, alcance ajustado curto demais, ou classes LCZ que na verdade não explicam
muito da variância para este evento específico) — nesse caso, prefira `lcz_interp_map_plus`
(interpolação por aprendizado de máquina com parâmetros de dossel urbano, notebook 05) em vez
da krigagem.

## Conclusão

Este notebook cobriu o fluxo de trabalho completo de interpolação geoestatística clássica:
quantificar a autocorrelação espacial com `lcz_variogram`, ver diretamente os três primitivos
de baixo nível de "ponto para grade" (`krige_predict`, `rbf_predict`, `idw_predict`), produzir
uma superfície de temperatura em grade completa com Krigagem com Deriva Externa sensível à LCZ
via `lcz_interp_map`, visualizá-la com `lcz_plot_interp` e — o mais importante — validar sua
precisão antes de confiar nela com `lcz_interp_eval`.

Isso se baseia diretamente em **03 · Ilha de Calor Urbana** (que comparou médias urbano vs.
rural no nível de estação) ao transformar essas comparações pontuais em uma superfície
contínua e defensável. Próximo na série: **05 · Interpolação por ML & Parâmetros de Dossel
Urbano** (`05_ml_interpolation_ucp`), que substitui o modelo geoestatístico por um modelo
Random Forest/OLS treinado em parâmetros morfológicos urbanos — uma escolha melhor quando a
densidade de estações é baixa demais para que as suposições baseadas em distância da krigagem
se sustentem.